## Import libraries

In [1]:
import scispacy
import spacy
from spacy import displacy
from scispacy.linking import EntityLinker

import warnings
warnings.filterwarnings("ignore")

Your CPU supports instructions that this binary was not compiled to use: SSE3 SSE4.1 SSE4.2 AVX AVX2
For maximum performance, you can install NMSLIB from sources 
pip install --no-binary :all: nmslib


## Extract entities using SciSpacy

In [2]:
text = "Microscopic observation of the pathology slide reveals sheets and nests of atypical squamous cells with abundant eosinophilic cytoplasm and significant nuclear atypia, characterized by increased nuclear size, irregular nuclear contours, and hyperchromasia. The architecture predominantly lacks keratinization, aligning with a non-keratinizing histological variant. Morphological assessment indicates poorly differentiated squamous cells with grade III differentiation, characterized by highly disorganized cell arrangements and pronounced pleomorphism. There is moderate tumor necrosis indicating areas of cell death, but no evidence of lymphatic, venous, or arterial invasion by the tumor. Additionally, non-neoplastic lung tissue exhibits signs of emphysema, pointing to alveolar damage and airspace enlargement. Based on these observations, the diagnosis is squamous cell carcinoma, non-keratinizing type, poorly differentiated (grade III)."

In [4]:
# Load Scispacy model
nlp = spacy.load("en_core_sci_lg")
nlp.add_pipe(
    "scispacy_linker",
    config={
        "resolve_abbreviations": True,
        "linker_name": "umls"
    }
)
linker = nlp.get_pipe("scispacy_linker")

In [5]:
# Process the report text with scispacy
doc = nlp(text)

for ent in doc.ents:
    print(ent)

Microscopic
observation
pathology slide
sheets
nests
atypical
squamous cells
eosinophilic cytoplasm
nuclear atypia
increased
nuclear size
irregular
nuclear contours
hyperchromasia
architecture
keratinization
non-keratinizing histological variant
Morphological
assessment
squamous cells
grade III differentiation
disorganized
pleomorphism
moderate
tumor necrosis
cell death
evidence
lymphatic
venous
arterial invasion
tumor
non-neoplastic
lung tissue
emphysema
alveolar damage
airspace enlargement
observations
diagnosis
squamous cell carcinoma
non-keratinizing type
poorly differentiated
grade III


In [24]:
AXES = {
    "histology": {
        "tuis": {"T191"},
        "keywords": ["carcinoma", "sarcoma", "neoplasm"]
    },
    "variant": {
        "tuis": {"T190", "T037", "T033"},
        "keywords": ["keratin", "non-keratin", "keratinizing"]
    },
    "grade": {
        "tuis": {"T080"},
        "keywords": ["poorly", "moderate", "well", "grade"]
    },
    "site": {
        "tuis": {"T023", "T029"},
        "keywords": ["lung", "pulmonary", "alveolar"]
    },
    "feature": {
        "tuis": {"T046", "T047"},
        "keywords": ["necrosis", "invasion", "pleomorphism"]
    }
}


In [25]:
from collections import defaultdict

axis_candidates = defaultdict(list)

for ent in doc.ents:
    ent_text_lower = ent.text.lower()

    for cui, score in ent._.kb_ents:
        entity = linker.kb.cui_to_entity[cui]
        entity_tuis = set(entity.types)

        for axis, cfg in AXES.items():
            # Check TUI compatibility
            if not entity_tuis.intersection(cfg["tuis"]):
                continue

            # Optional keyword boost (light heuristic)
            keyword_boost = 0.0
            if any(k in ent_text_lower for k in cfg["keywords"]):
                keyword_boost = 0.05

            axis_candidates[axis].append({
                "cui": cui,
                "text": ent.text,
                "canonical_name": entity.canonical_name,
                "score": score + keyword_boost
            })


In [26]:
final_result = {}

for axis, candidates in axis_candidates.items():
    print(f"Axis: {axis}")
    if not candidates:
        final_result[axis] = None
        continue

    # Deduplicate by CUI, keep highest score per CUI
    best_per_cui = {}
    for c in candidates:
        if c["cui"] not in best_per_cui or c["score"] > best_per_cui[c["cui"]]["score"]:
            best_per_cui[c["cui"]] = c

    # Select highest scoring concept for this axis
    best_entity = max(best_per_cui.values(), key=lambda x: x["score"])

    final_result[axis] = {
        "text": best_entity["text"],
        "canonical_name": best_entity["canonical_name"],
        "cui": best_entity["cui"],
        "score": round(best_entity["score"], 4)
    }


Axis: grade
Axis: variant
Axis: feature
Axis: histology
Axis: site


In [ ]:
print("\nFinal axis-normalized entities:\n")
for axis, value in final_result.items():
    print(f"{axis}: {value}")


Final axis-normalized entities:

grade: {'text': 'grade III', 'canonical_name': 'Poorly differentiated', 'cui': 'C0205617', 'score': 1.0412}
variant: {'text': 'pleomorphism', 'canonical_name': 'Pleomorphism (morphologic abnormality)', 'cui': 'C0032219', 'score': 0.9922}
feature: {'text': 'emphysema', 'canonical_name': 'Pathological accumulation of air in tissues', 'cui': 'C0013990', 'score': 0.9892}
histology: {'text': 'squamous cell carcinoma', 'canonical_name': 'Squamous cell carcinoma', 'cui': 'C0007137', 'score': 1.0236}
site: {'text': 'lymphatic', 'canonical_name': 'Lymphatic vessel', 'cui': 'C0229889', 'score': 0.9814}
